# Generative D-peptide design: full workflow with an hIAPP case study

This primary notebook provides a step-by-step, auditable implementation of a reusable structure-guided reverse-D peptide design framework. It centralizes model installation, target-conditioned all-atom generation, inverse folding, stereochemical conversion, complex prediction, retry recovery, scoring, screening, and structure-review commands for a Slurm server with NVIDIA GPUs. CPU data-processing and GPU submission switches make every stage inspectable before execution. The parameter cells can be adapted to a new target; hIAPP is the fully documented reference campaign.

Workflow: 9ULZ input → 2,000 BoltzGen backbone designs → four BoltzIF sequences per backbone (temperature 0.2; exclude C) → deduplication and reverse-D SMILES → Boltz-2 structure/affinity prediction → retry failed jobs → merge scores → sequence physicochemical descriptors → four-layer prioritization → manual structure review.

## 0. Reproducibility design and interpretation

- **Framework scope:** replace the target structure, retained residues, binding-site definition, peptide constraints, and assay plan to initiate a new campaign. The hIAPP values below reproduce the deposited case study.
- **Chirality boundary:** BoltzGen generates an L-sequence/structure hypothesis; this project then applies reverse-D conversion and explicit stereochemical SMILES. The retro-inverso transformation is tested computationally but ultimately requires experimental validation.
- BoltzGen and Boltz use separate environments. Installation commands follow their official GitHub repositories. Record the exact package versions, checkpoint paths, and SHA256 values for every production run.
- Boltz documents its affinity module for protein-small-molecule prediction, discourages ligands above 56 atoms, and imposes a 128-atom hard limit. This project encodes reverse-D peptides as stereochemical ligand SMILES as an extension for relative ranking. The affinity output must not be reported as an experimental binding constant.
- Predicted structures and scores support candidate prioritization; they do not constitute experimental evidence of activity, selectivity, or mechanism.

In [ ]:
from pathlib import Path
import os, shlex, subprocess, textwrap
import pandas as pd

# REPO_ROOT needs no change when Jupyter is started from the repository root.
REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == 'notebooks':
    REPO_ROOT = REPO_ROOT.parent

# Edit only this block: server project path, environment names, and model-data paths.
SERVER_ROOT = Path('/data/run01/USER/XMY')
CONDA_SH = '/opt/conda/etc/profile.d/conda.sh'
BOLTZGEN_ENV = 'boltzgen'
BOLTZ_ENV = 'boltz'
BOLTZGEN_MOLDIR = Path('/data/run01/USER/models/boltzgen/mols')
BOLTZGEN_DESIGN_CHECKPOINT = Path('/data/run01/USER/models/boltzgen/boltzgen1_diverse.ckpt')
BOLTZGEN_IFOLD_CHECKPOINT = 'huggingface:boltzgen/boltzgen1_ifold:boltzgen1_ifold.ckpt'
BOLTZ_CACHE = Path('/data/run01/USER/models/boltz_cache')

DESIGN_RUN = SERVER_ROOT / 'outputs' / 'hIAPP_batch1'
IFOLD_RUN = SERVER_ROOT / 'outputs' / 'hIAPP_batch1_ifold'
WORK = SERVER_ROOT / 'summary' / 'full_reproduction'
INPUT_ROOT = SERVER_ROOT / 'input'
BOLTZ_OUTPUTS = SERVER_ROOT / 'outputs_boltz2'

# Safety switches: keep False until every printed path and command has been reviewed.
RUN_CPU_STEPS = False
SUBMIT_GPU_STEPS = False

def shell(command, execute=False):
    command = textwrap.dedent(command).strip()
    print('$ ' + command.replace('\n', '\n  '))
    if execute:
        subprocess.run(command, shell=True, check=True, executable='/bin/bash', cwd=REPO_ROOT)

print('Repository:', REPO_ROOT)
print('Server root:', SERVER_ROOT)

## 1. Install the two model environments using official instructions

Official resources: [BoltzGen README](https://github.com/HannesStark/boltzgen), [Boltz README](https://github.com/jwohlwend/boltz), and the [Boltz prediction guide](https://github.com/jwohlwend/boltz/blob/main/docs/prediction.md).

BoltzGen requires Python 3.11 or newer and shows a Python 3.12 environment in its official example. Boltz recommends `pip install boltz[cuda] -U`. Run the displayed commands on the configured GPU server and preserve the exported environment files with the computational results.

In [ ]:
print('''# BoltzGen (official PyPI installation)
conda create -n boltzgen python=3.12 -y
conda run -n boltzgen python -m pip install -U boltzgen
conda run -n boltzgen boltzgen --help

# Boltz (official CUDA installation)
conda create -n boltz python=3.11 -y
conda run -n boltz python -m pip install -U 'boltz[cuda]'
conda run -n boltz boltz predict --help

# Reproducibility record: save exact environments immediately after installation
conda run -n boltzgen python -m pip freeze > environment_boltzgen.txt
conda run -n boltz python -m pip freeze > environment_boltz.txt
nvidia-smi > nvidia-smi.txt
''')

## 2. Create the server directory layout and inspect inputs

`configs/boltzgen/9ULZ.yaml` retains residues 19–37 of chain D and defines residues 21–37 as the binding region. The Boltz-2 template uses the same hIAPP fragment, a fixed structural template, and a ligand-SMILES placeholder.

In [ ]:
for relative in ['configs/boltzgen/9ULZ.yaml', 'configs/boltz2/hIAPP_template.yaml']:
    path = REPO_ROOT / relative
    print(f'--- {relative} ---')
    print(path.read_text())

shell(f"mkdir -p {SERVER_ROOT}/{{input,outputs,outputs_boltz2,logs,tmp,summary}} {WORK}", RUN_CPU_STEPS)

## 3. BoltzGen de novo backbone design (2,000 designs)

This stage preserves the recorded `peptide-anything` protocol, design-only mode, 2,000 requested designs, diffusion batch size 10, and custom molecule/checkpoint paths. Validate the input with `boltzgen check`, then run the production job through Slurm. Set `BOLTZGEN_REUSE=1` when resuming an interrupted run.

In [ ]:
design_submit = f'''
export REPO_ROOT={shlex.quote(str(REPO_ROOT))}
export CONDA_SH={shlex.quote(CONDA_SH)}
export CONDA_ENV_BOLTZGEN={shlex.quote(BOLTZGEN_ENV)}
export BOLTZGEN_OUTPUT={shlex.quote(str(DESIGN_RUN))}
export BOLTZGEN_MOLDIR={shlex.quote(str(BOLTZGEN_MOLDIR))}
export BOLTZGEN_DESIGN_CHECKPOINT={shlex.quote(str(BOLTZGEN_DESIGN_CHECKPOINT))}
export NUM_DESIGNS=2000
export DIFFUSION_BATCH_SIZE=10
sbatch hpc/run_boltzgen_design.slurm
'''
shell(design_submit, SUBMIT_GPU_STEPS)

In [ ]:
# Post-run check; the recorded run produced 1,990 completed design CIF/NPZ pairs.
shell(f"squeue -u $USER; find {DESIGN_RUN}/intermediate_designs -maxdepth 1 -name '*.cif' | wc -l")

## 4. Extract and deduplicate design sequences

The script reads `_entity_poly.pdbx_seq_one_letter_code` from each CIF. The first entity is the designed peptide and the second is fixed hIAPP. The retained raw data verify 1,990 CIF files → 1,956 unique sequences.

In [ ]:
design_csv = WORK / 'batch1_design.csv'
shell(f'''python scripts/extract_cif_sequences.py \
  {DESIGN_RUN}/intermediate_designs \
  {design_csv}''', RUN_CPU_STEPS)

## 5. BoltzIF sequence redesign

The top-level BoltzGen CLI exposes sequence multiplicity and excluded residues but does not expose the required sampling temperature as a separate option. The repository script therefore derives a run configuration from the installed package's official `inverse_fold.yaml`, overrides only the recorded settings (`multiplicity=4`, `temperature=0.2`, and `avoid=C`), and then calls the installed package entry point. Every CIF requires a same-stem NPZ file to preserve the design mask.

In [ ]:
ifold_submit = f'''
export REPO_ROOT={shlex.quote(str(REPO_ROOT))}
export CONDA_SH={shlex.quote(CONDA_SH)}
export CONDA_ENV_BOLTZGEN={shlex.quote(BOLTZGEN_ENV)}
export BOLTZGEN_OUTPUT={shlex.quote(str(DESIGN_RUN))}
export BOLTZGEN_IFOLD_OUTPUT={shlex.quote(str(IFOLD_RUN))}
export BOLTZGEN_MOLDIR={shlex.quote(str(BOLTZGEN_MOLDIR))}
export BOLTZGEN_IFOLD_CHECKPOINT={shlex.quote(BOLTZGEN_IFOLD_CHECKPOINT)}
export NUM_GPUS=4
export NUM_WORKERS=4
sbatch hpc/run_boltzgen_inverse_folding.slurm
'''
shell(ifold_submit, SUBMIT_GPU_STEPS)

## 6. Merge design and BoltzIF sequences and generate reverse-D SMILES

Retained raw results: 7,960 BoltzIF CIF files → 7,854 unique sequences. Merging these with 1,956 unique design sequences gives 9,808 unique sequences. `seq_rev` reverses the original L-sequence and writes it in lower case. Lower case is only a data convention; D stereochemistry is encoded explicitly in the SMILES.

In [ ]:
ifold_csv = WORK / 'batch1_IF_02.csv'
merged_csv = WORK / 'merged_designs.csv'
reversed_csv = WORK / 'merged_designs_reversed.csv'
smiles_csv = WORK / 'merged_designs_reversed_smiles.csv'
shell(f'''python scripts/extract_cif_sequences.py \
  {IFOLD_RUN}/intermediate_designs_inverse_folded {ifold_csv}
python scripts/merge_design_batches.py \
  --input {design_csv}=design \
  --input {ifold_csv}=IF_02 \
  --output {merged_csv}
python scripts/reverse_sequences.py {merged_csv} {reversed_csv}
python scripts/peptide_to_smiles.py {reversed_csv} {smiles_csv}''', RUN_CPU_STEPS)

In [ ]:
# Automatically verify the recorded milestones.
if RUN_CPU_STEPS:
    d = pd.read_csv(design_csv); i = pd.read_csv(ifold_csv); m = pd.read_csv(smiles_csv)
    print({'design_raw': int(d['count'].sum()), 'design_unique': len(d),
           'ifold_raw': int(i['count'].sum()), 'ifold_unique': len(i), 'merged_unique': len(m)})
    assert (int(d['count'].sum()), len(d)) == (1990, 1956)
    assert (int(i['count'].sum()), len(i)) == (7960, 7854)
    assert len(m) == 9808

## 7. Generate Boltz-2 input YAML files

Batch 1 contains 9,808 AI designs. Batch 3 contains the 76 reference-inhibitor entries with usable SMILES. The template fixes the hIAPP sequence and the 9ULZ chain-D structure and represents the D-peptide as ligand X.

In [ ]:
batch1_yaml = INPUT_ROOT / 'batch1_yaml'
batch3_yaml = INPUT_ROOT / 'batch3_yaml'
reference_clean = WORK / 'reference_inhibitors_smiles_76.csv'
if RUN_CPU_STEPS:
    ref = pd.read_csv(REPO_ROOT / 'data/designs/reference_inhibitors_smiles.csv')
    ref = ref.dropna(subset=['seq_rev', 'smiles']).copy()
    ref.to_csv(reference_clean, index=False, encoding='utf-8-sig')
    assert len(ref) == 76
shell(f'''python scripts/make_boltz_yaml.py \
  {smiles_csv} configs/boltz2/hIAPP_template.yaml {batch1_yaml} \
  --template-cif {REPO_ROOT}/configs/boltz2/9ULZ.cif
python scripts/make_boltz_yaml.py \
  {reference_clean} configs/boltz2/hIAPP_template.yaml {batch3_yaml} \
  --template-cif {REPO_ROOT}/configs/boltz2/9ULZ.cif''', RUN_CPU_STEPS)

## 8. Boltz-2 smoke test and parallel prediction

Test one YAML first. After it succeeds, use the bounded dispatcher to maintain no more than eight jobs with the same name, checking every 20 seconds. The portable execution chain is `submit_keep8.sh → run_lig_array_large.slurm → run_lig_one_large.sh`; all paths, environment names, caches, resources, and concurrency settings are configurable.

In [ ]:
boltz_env = f'''
export REPO_ROOT={shlex.quote(str(REPO_ROOT))}
export PROJECT_ROOT={shlex.quote(str(SERVER_ROOT))}
export CONDA_SH={shlex.quote(CONDA_SH)}
export CONDA_ENV={shlex.quote(BOLTZ_ENV)}
export BOLTZ_CACHE={shlex.quote(str(BOLTZ_CACHE))}
export BOLTZ_OUTPUTS={shlex.quote(str(BOLTZ_OUTPUTS))}
'''
print('Boltz output root:', BOLTZ_OUTPUTS)
print('Smoke-test command:')
print(boltz_env + f"sbatch --array=1-1%1 hpc/run_lig_array_large.slurm $(find {batch1_yaml} -name '*.yaml' | sort | head -n 1)")

In [ ]:
batch1_dispatch = boltz_env + f'''
nohup bash hpc/submit_keep8.sh \
  {batch1_yaml} 8 20 {batch1_yaml}/dispatch_state \
  {REPO_ROOT}/hpc/run_lig_array_large.slurm boltz_arr 1 \
  > {SERVER_ROOT}/logs/keep8_batch1.log 2>&1 &
'''
shell(batch1_dispatch, SUBMIT_GPU_STEPS)

In [ ]:
# Read-only monitoring commands; these do not change job state.
print(f'''squeue -u $USER
tail -n 100 {SERVER_ROOT}/logs/keep8_batch1.log
cat {batch1_yaml}/dispatch_state/next_index
tail -n 20 {batch1_yaml}/dispatch_state/submitted.tsv
find {BOLTZ_OUTPUTS} -name '*_model_0.cif' | wc -l''')

## 9. Retry failed predictions (batch 2)

Scan for tasks without `*_model_0.cif` only after dispatch has finished and no related jobs remain in the queue. Do not create batch 2 while predictions are still running. In the recorded run, the first 9,808 tasks produced 9,374 successful output directories, and 438 inputs were collected for retry.

In [ ]:
batch2_yaml = INPUT_ROOT / 'batch2_yaml'
shell(f"python scripts/find_missing_model0.py {BOLTZ_OUTPUTS} {batch1_yaml} {batch2_yaml}", RUN_CPU_STEPS)
print(boltz_env + f'''nohup bash hpc/submit_keep8.sh {batch2_yaml} 8 20 \
  {batch2_yaml}/dispatch_state {REPO_ROOT}/hpc/run_lig_array_large.slurm boltz_arr 1 \
  > {SERVER_ROOT}/logs/keep8_batch2.log 2>&1 &''')

## 10. Reference-inhibitor prediction (batch 3)

Batch 3 uses the same model settings to provide a consistent computational reference for natural and previously reported molecules. Complex modifications are recorded explicitly in the SMILES table; entries that cannot be standardized must not be inferred silently.

In [ ]:
print(boltz_env + f'''nohup bash hpc/submit_keep8.sh {batch3_yaml} 8 20 \
  {batch3_yaml}/dispatch_state {REPO_ROOT}/hpc/run_lig_array_large.slurm boltz_arr 1 \
  > {SERVER_ROOT}/logs/keep8_batch3.log 2>&1 &''')

## 11. Collect model outputs, merge design tables, and calculate transparent physicochemical descriptors

After all batches finish, collect model-0 confidence files and matching affinity JSON files. Later retry results supersede earlier results for the same ID. The physicochemical properties are documented sequence-level heuristics used only for within-pool ranking; see `docs/PHYSICOCHEMICAL.md` and `docs/REFERENCES.md`.

In [ ]:
boltz_scores = WORK / 'boltz_scores_all.csv'
merged_all = WORK / 'merged_all.csv'
merged_scored = WORK / 'merged_all_scored.csv'
shell(f'''python scripts/collect_boltz_scores.py \
  --outputs-dir {BOLTZ_OUTPUTS} --output {boltz_scores}
python scripts/merge_design_and_scores.py \
  --design {smiles_csv} --design {reference_clean} \
  --scores {boltz_scores} --output {merged_all}
python scripts/score_sequence_properties.py {merged_all} {merged_scored}''', RUN_CPU_STEPS)

## 12. Four-layer prioritization

Hard gate: complete affinity heads, `min(pTM, ipTM) ≥ 0.65`, solubility tendency ≥ 40, and aggregation risk ≤ 65. Next, recompute affinity consensus, structure quality, and developability to select P1/P2/P3. The separate P4 branch prioritizes 11–12-aa peptides with `min(pTM, ipTM) ≥ 0.75`. Exact equations are documented in `docs/METHODS.md`.

In [ ]:
screen_dir = WORK / 'screening'
shell(f"python scripts/screen_candidates.py {merged_scored} {screen_dir}", RUN_CPU_STEPS)
if RUN_CPU_STEPS:
    hard = pd.read_csv(screen_dir / 'hard_gate_904.csv')
    top32 = pd.read_csv(screen_dir / 'p1_p2_p3_top32.csv')
    p4 = pd.read_csv(screen_dir / 'p4_top10.csv')
    print('Hard gate:', len(hard))
    print('P1/P2/P3:', top32['priority'].value_counts().sort_index().to_dict())
    print('P4 review pool:', len(p4))

## 13. Structure review and final set of 12

Automated screening produces a 32-member P1–P3 pool and a 10-member P4 pool for structure review. Final selection is manual: inspect occupancy of the intended residues 21–37, interface contacts and hydrogen bonds, obvious steric clashes, extended conformations, sequence redundancy, and synthetic tractability. Record a reason for every inclusion or exclusion so visual judgment is not misrepresented as an automated score. The recorded result was P1 × 8, P2 × 1, P3 × 1, and P4 × 2. A new stochastic generation run may yield different candidates; do not automatically reuse the historical final 12.

In [ ]:
review_csv = WORK / 'structure_review.csv'
fresh_final = WORK / 'final_12_after_review.csv'
shell(f'''python scripts/prepare_structure_review.py \
  {screen_dir}/p1_p2_p3_top32.csv {screen_dir}/p4_top10.csv {review_csv}''', RUN_CPU_STEPS)
print('Review every structure in PyMOL and complete:', review_csv)
print('Then validate and export the completed review:')
print(f'python scripts/finalize_review_selection.py {review_csv} {fresh_final}')
print('\nHistorical set of 12 (reproduction reference only):')
historical = pd.read_csv(REPO_ROOT / 'results/final_candidates/final_12.csv')
display(historical)
assert len(historical) == 12

In [ ]:
# Collect all model-0 CIF files and write a traceable manifest for PyMOL review.
collected_cif = WORK / 'collected_model0_cif'
shell(f"python scripts/collect_model0_cif.py {BOLTZ_OUTPUTS} {collected_cif}", RUN_CPU_STEPS)

## 14. Archive the reproducibility evidence

Preserve both `pip freeze` files, GPU/Slurm logs, the generated BoltzGen configuration, the input-YAML manifest, every JSON/CIF path, the failed-job retry list, score tables, structure-review reasons, and the code commit. Do not place large model caches or complete raw-output trees in ordinary Git history. Use a GitHub Release, Git LFS, or a data repository and record checksums plus download locations here.

In [ ]:
print(f'''# Example: archive computational evidence on the server
find {INPUT_ROOT} -type f -name '*.yaml' -print0 | sort -z | xargs -0 sha256sum > {WORK}/input_yaml_sha256.txt
find {WORK} -maxdepth 3 -type f -print0 | sort -z | xargs -0 sha256sum > {WORK}/analysis_sha256.txt
tar -czf hiapp_reproduction_logs_and_tables.tar.gz -C {SERVER_ROOT} logs summary
''')